# Test filter_service and image_filters

In [4]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: C:\Users\T22127A\Documents\priv\Uni\priv_dev\dermascan


In [5]:
from preprocessing import image_filters
from preprocessing.filter_service import FilterService
from dataset_manager import DatasetManager 

In [7]:
filter_service = FilterService(
    data_dir=str(proj_dir / "data")
)

dataset_manager = DatasetManager(
    data_dir=str(proj_dir / "data")
)

### Test applying and saving filters

#### Apply all filters for single image

In [26]:
res = filter_service.apply_texture_filters(image_id="ISIC_0024306", filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter])
res['lbp_filter'].shape


(450, 600, 3)

In [ ]:
filter_service.save_applied_filters(res, image_id="ISIC_0024306")

#### Apply filters to multiple images and save them

In [17]:
dataset_manager.load_cleaned_metadata(
    filter_melanoma_only=True,
)
dataset_manager.df['dx'].value_counts()

dx
nv     6703
mel    1113
Name: count, dtype: int64

In [23]:
nv_samples = dataset_manager.get_n_samples(1113, lesion_type='nv')
mel_samples = dataset_manager.get_n_samples(1113, lesion_type='mel')

sample_df = pd.concat([nv_samples, mel_samples]) # concat
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

sample_df['dx'].value_counts()

dx
nv     1113
mel    1113
Name: count, dtype: int64

In [ ]:
result_map = filter_service.filtered_image_generator(
    list_image_ids=sample_df['image_id'].tolist(),
    filter_list=[image_filters.glcm_filter, image_filters.gabor_filter, image_filters.lbp_filter],
    batch_size=10,
    asnpz=False
)

In [27]:
filter_service.save_applied_filters_batch(
    filters_results_map=result_map
)

### Load filtered image

In [ ]:
img = np.load(str(proj_dir / "data" / "filtered" / "ISIC_0028215.npz"))

In [ ]:
print(img['gabor_filter'])

[[0.00119036 0.00040971 0.00226006 ... 0.00498912 0.00627132 0.00472366]
 [0.00091862 0.00064804 0.00269534 ... 0.00533996 0.00580251 0.00537885]
 [0.00075732 0.00072194 0.00221816 ... 0.00587587 0.00534302 0.00597247]
 ...
 [0.0058884  0.00277858 0.0059192  ... 0.00290131 0.00102515 0.0003759 ]
 [0.00562135 0.00398375 0.0051641  ... 0.00319549 0.0011711  0.00082788]
 [0.0052011  0.00530117 0.0047166  ... 0.00264601 0.00107729 0.00143592]]
